## Checks de estructura — EPA

### Lectura y estructura básica
- ☐ Verificar `shape` (filas y columnas)
- ☐ Revisar nombres de columnas 
- ☐ Revisar tipos de dato 
- ☐ Visualizar primeras filas 

### Calidad de datos
- ☐ Comprobar valores nulos 
- ☐ Verificar que no hay columnas vacías
- ☐ Confirmar lectura correcta (separador / encoding)
- ☐ Revisar que no hay columnas mal parseadas

### Claves e identificadores
- ☐ Verificar existencia de claves (`CCLO`, `CCAA`, `NVIV`, `NPER`)
- ☐ Comprobar que no hay nulos en claves
- ☐ Revisar tipos de las claves (consistentes)
- ☐ Detectar duplicados en claves

### Consistencia entre ficheros
- ☐ Comparar columnas entre datasets
- ☐ Detectar columnas faltantes
- ☐ Detectar columnas extra
- ☐ Verificar consistencia de tipos entre trimestres
- ☐ Confirmar consistencia de nombres de variables

### Pesos (anexo)
- ☐ Verificar existencia de `FACB2021` en anexo
- ☐ Confirmar ausencia (o no uso) del peso en el principal
- ☐ Revisar tipos de la variable de peso

### Pre-merge
- ☐ Comparar número de claves únicas (principal vs anexo)
- ☐ Verificar que las claves casan estructuralmente

### Post-merge
- ☐ Verificar cobertura del merge
- ☐ Comprobar nulos en `FACB2021`
- ☐ Validar número de filas tras el merge
- ☐ Confirmar que no hay duplicaciones inesperadas

### Consistencia temporal (2021–2025)
- ☐ Verificar mismas columnas en todos los trimestres
- ☐ Detectar cambios de esquema
- ☐ Confirmar estabilidad de claves

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path # esto es para simplificar el tema de las rutas

In [2]:
os.getcwd()

'/home/clara/Documentos/NLA/proyecto_edu/notebooks'

In [3]:
# Esta es la primera carga rudimentaria que hice, después pensé en hacerla un poco
# más apañada

# # 2021
# t12021 = pd.read_csv("../data/raw/EPA_2021T1.csv")
# t22021 = pd.read_csv("../data/raw/EPA_2021T2.csv")
# t32021 = pd.read_csv("../data/raw/EPA_2021T3.csv")
# t42021 = pd.read_csv("../data/raw/EPA_2021T4.csv")
# # 2022
# t12022 = pd.read_csv("../data/raw/EPA_2022T1.csv")
# t22022 = pd.read_csv("../data/raw/EPA_2022T2.csv")
# t32022 = pd.read_csv("../data/raw/EPA_2022T3.csv")
# t42022 = pd.read_csv("../data/raw/EPA_2022T4.csv")
# # 2023
# t12023 = pd.read_csv("../data/raw/EPA_2023T1.csv")
# t22023 = pd.read_csv("../data/raw/EPA_2023T2.tab", sep="\t")
# t32023 = pd.read_csv("../data/raw/EPA_2023T3.tab", sep="\t")
# t42023 = pd.read_csv("../data/raw/EPA_2023T4.tab", sep="\t")
# # 2024
# t12024 = pd.read_csv("../data/raw/EPA_2024T1.tab", sep="\t")
# t22024 = pd.read_csv("../data/raw/EPA_2024T2.tab", sep="\t")
# t32024 = pd.read_csv("../data/raw/EPA_2024T3.tab", sep="\t")
# t42024 = pd.read_csv("../data/raw/EPA_2024T4.tab", sep="\t")
# # 2025
# t12025 = pd.read_csv("../data/raw/EPA_2025T1.tab", sep="\t")
# t22025 = pd.read_csv("../data/raw/EPA_2025T2.tab", sep="\t")
# t32025 = pd.read_csv("../data/raw/EPA_2025T3.tab", sep="\t")
# t42025 = pd.read_csv("../data/raw/EPA_2025T4.tab", sep="\t")

### Cargar los archivos generales

In [4]:
ruta = Path("../data/raw") # así establecemos dónde está la carpeta con los datos

dfs = {} # aquí creamos un diccionario donde vamos a guardar los df

for file in sorted(ruta.glob("EPA_*")): # glob() sirve para buscar archivos que cumplen un patrón
                                        # "EPA_*" busca todo lo que empieza por EPA_

    # aquí vamos a identificar por extensión, ya que tenemos dos diferentes
    # y tenemos que tratarlas de forma diferente para que funciona

    if file.suffix ==".tab":
        df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
    else:
        df = pd.read_csv(file, sep= "\t", encoding="utf-8", low_memory=False)
        # esto lo he metido como else porque la extensión es diferente, pero 
        # en realidad también separa con tabulación; lo mantenemos así por si
        # acaso da algún problema

    # vamos a usar el nombre del fichero como clave del diccionario

    key = file.stem # por ejemplo, quedaría EPA_2021T1; stem = nombre base del archivo

    dfs[key] = df # y con esto metemos el dataframe en el diccionario

In [5]:
dfs["EPA_2021T1"].head().T

,0,1,2,3,4
CICLO,194,194,194,194,194
CCAA,16,16,16,16,16
PROV,1,1,1,1,1
NVIVI,1,1,2,2,2
NIVEL,1,1,1,1,2
...,...,...,...,...,...
DAUSOTR,NaN,NaN,NaN,NaN,NaN
TRAANT,1,1,1,1,
AOI,04,03,04,04,
RELLB4,,,,,


### Cargar los archivos de los anexos

In [6]:
ruta = Path("../data/raw") # así establecemos dónde está la carpeta con los datos

dfs_a = {} # aquí creamos un diccionario donde vamos a guardar los df de los anexos

for file in sorted(ruta.glob("EPAAnexo_*")): # glob() sirve para buscar archivos que cumplen un patrón
                                        # "EPA_*" busca todo lo que empieza por EPA_

    # aquí vamos a identificar por extensión, ya que tenemos dos diferentes
    # y tenemos que tratarlas de forma diferente para que funcione

    if file.suffix ==".tab":
        df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
    else:
        df = pd.read_csv(file, encoding="utf-8", low_memory=False)

    # vamos a usar el nombre del fichero como clave del diccionario

    key = file.stem # por ejemplo, quedaría EPA_2021T1; stem = nombre base del archivo

    dfs_a[key] = df # y con esto metemos el dataframe en el diccionario

In [7]:
dfs_a["EPAAnexo_2021T1"].head().T

,0,1,2,3,4
CICLO,194.00,194.00,194.00,194.00,194.00
CCAA,16.00,16.00,16.00,16.00,16.00
NVIVI,1.00,1.00,2.00,2.00,2.00
NIVEL,1.00,1.00,1.00,1.00,2.00
NPERS,1.00,2.00,1.00,2.00,3.00
FACB2021,174.32,174.32,225.37,225.37,225.37


In [8]:
dfs["EPA_2021T1"].columns

Index(['CICLO', 'CCAA', 'PROV', 'NVIVI', 'NIVEL', 'NPERS', 'EDAD1', 'RELPP1',
       'SEXO1', 'NCONY', 'NPADRE', 'NMADRE', 'RELLMILI', 'ECIV1', 'PRONA1',
       'REGNA1', 'NAC1', 'EXREGNA1', 'ANORE1', 'NFORMA', 'RELLB1', 'EDADEST',
       'CURSR', 'NCURSR', 'CURSNR', 'OBJFORM', 'RELLB2', 'TRAREM', 'AYUDFA',
       'AUSENT', 'RZNOTB', 'VINCUL', 'NUEVEM', 'OCUP1', 'ACT1', 'SITU', 'SP',
       'DUCON1', 'DUCON2', 'DUCON3', 'TCONTM', 'TCONTD', 'DREN', 'DCOM',
       'PROEST', 'REGEST', 'PARCO1', 'PARCO2', 'HORASP', 'HORASH', 'HORASE',
       'EXTRA', 'EXTPAG', 'EXTNPG', 'RELLB3', 'TRAPLU', 'OCUPLU1', 'ACTPLU1',
       'SITPLU', 'HORPLU', 'MASHOR', 'DISMAS', 'RZNDISH', 'HORDES', 'BUSOTR',
       'BUSCA', 'DESEA', 'FOBACT', 'NBUSCA', 'RZULT', 'ITBU', 'DISP', 'RZNDIS',
       'EMPANT', 'DTANT', 'OCUPA', 'ACTA', 'SITUA', 'OFEMP', 'SIDI1', 'SIDI2',
       'SIDI3', 'SIDAC1', 'SIDAC2', 'DAUSVAC', 'DAUSENF', 'DAUSOTR', 'TRAANT',
       'AOI', 'RELLB4', 'FACTOREL'],
      dtype='object')

In [9]:
lista_columnas = []
for e in dfs.values():
    c= set(e.columns)
    lista_columnas.append(c)

display(lista_columnas)

[{'ACT1',
  'ACTA',
  'ACTPLU1',
  'ANORE1',
  'AOI',
  'AUSENT',
  'AYUDFA',
  'BUSCA',
  'BUSOTR',
  'CCAA',
  'CICLO',
  'CURSNR',
  'CURSR',
  'DAUSENF',
  'DAUSOTR',
  'DAUSVAC',
  'DCOM',
  'DESEA',
  'DISMAS',
  'DISP',
  'DREN',
  'DTANT',
  'DUCON1',
  'DUCON2',
  'DUCON3',
  'ECIV1',
  'EDAD1',
  'EDADEST',
  'EMPANT',
  'EXREGNA1',
  'EXTNPG',
  'EXTPAG',
  'EXTRA',
  'FACTOREL',
  'FOBACT',
  'HORASE',
  'HORASH',
  'HORASP',
  'HORDES',
  'HORPLU',
  'ITBU',
  'MASHOR',
  'NAC1',
  'NBUSCA',
  'NCONY',
  'NCURSR',
  'NFORMA',
  'NIVEL',
  'NMADRE',
  'NPADRE',
  'NPERS',
  'NUEVEM',
  'NVIVI',
  'OBJFORM',
  'OCUP1',
  'OCUPA',
  'OCUPLU1',
  'OFEMP',
  'PARCO1',
  'PARCO2',
  'PROEST',
  'PRONA1',
  'PROV',
  'REGEST',
  'REGNA1',
  'RELLB1',
  'RELLB2',
  'RELLB3',
  'RELLB4',
  'RELLMILI',
  'RELPP1',
  'RZNDIS',
  'RZNDISH',
  'RZNOTB',
  'RZULT',
  'SEXO1',
  'SIDAC1',
  'SIDAC2',
  'SIDI1',
  'SIDI2',
  'SIDI3',
  'SITPLU',
  'SITU',
  'SITUA',
  'SP',
  'TCONTD',
  

In [10]:
i = 0
ref = lista_columnas[0]

for c in lista_columnas:
    i +=1
    if c == lista_columnas[0]:
        print(f"El grupo {i} es igual, todo bien.")
    else:
        faltan = ref - c
        sobran = c - ref
        print(f"En el grupo{i} faltan estas columnas: \n{faltan}\n y sobran estas columnas \n{sobran}\n")
              

El grupo 1 es igual, todo bien.
El grupo 2 es igual, todo bien.
El grupo 3 es igual, todo bien.
El grupo 4 es igual, todo bien.
El grupo 5 es igual, todo bien.
El grupo 6 es igual, todo bien.
El grupo 7 es igual, todo bien.
El grupo 8 es igual, todo bien.
El grupo 9 es igual, todo bien.
En el grupo10 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el grupo11 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el grupo12 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el grupo13 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el grupo14 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el grupo15 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el grupo16 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el grupo17 faltan estas columnas: 
{'HORPLU'}
 y sobran estas columnas 
{'HOREPLU'}

En el gr

- COLUMNAS: `HORPLU` vs `HOREPLU` -> normalizar como `HOREPLU`
- Debe normalizarse en src/data/clean.py